# Deploy Qwen3.8-27B on SageMaker Endpoint using AWS Python API (boto3)


### [Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B) is the dense 27B member of the Qwen3.8 family. Built on the Qwen3.5 architecture (hybrid Gated DeltaNet + Gated Attention with a built-in multi-token prediction head), it is a native vision-language model that understands text, images, and video.

#### Qwen3.8 Highlights

- *Core Capabilities*: comprehensive improvements across coding, professional work, research, and long-horizon agentic tasks.

- *Agent Execution*: stronger autonomous planning and better handling of environment feedback, leading to more reliable end-to-end task completion.

- *Flexible Thinking Control*: thinking mode is on by default and can be disabled per request; reasoning depth can be tuned with `reasoning_effort` (`xhigh` default, `medium`, `low`), and reasoning context from historical messages is retained via `preserve_thinking`.

- *Vision-Language Understanding*: native support for image and video understanding.

- *Context Length*: 262,144 tokens natively, extensible up to 1M with YaRN.

Please refer to the official [model card](https://huggingface.co/Qwen/Qwen3.8-27B) for more details.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
# Tested on ml.g6e.24xlarge (4x L40S). ml.g6e.12xlarge has the same 4x L40S GPUs and should take the same settings,
# but it was not tested: it could not be provisioned (InsufficientInstanceCapacity) during testing.
instance = {"type": "ml.g6e.24xlarge", "num_gpu": 4}
model_id = "Qwen/Qwen3.8-27B"
model_name = f"qwen38-27b-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 1800
variant_name = "v1"

##### Amazon SageMaker AI makes model deployment process very easy. The deployment steps are exactly the same regardless of model framework container you choose. Below we show you 3 options:
1. DLC vLLM (0.32.0)
2. DLC SGLang (0.5.22)
3. SageMaker LMI (v30)

#### Please chose one option and run only the cell for your container of choice

#### We will be using native Qwen3.8 multi-token prediction (MTP). The net effect is higher throughput (more tokens per second) with no degradation in output quality, since the verification step ensures the final output matches what standard autoregressive decoding would produce. It's essentially "free" speculation baked into the model itself, removing the overhead of training and serving a separate smaller draft model.

### Option 1: vLLM

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.32.0-gpu-py312-cu130-ubuntu24.04-sagemaker"

spec_config = {
    "method": "mtp",
    "num_speculative_tokens": 3
}

common_env = {
    "HF_MODEL_ID": model_id,
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "262144",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_VLLM_REASONING_PARSER": "qwen3",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | vllm_env

### Option 2: SGLang

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:0.5.22-gpu-py312-cu130-ubuntu24.04-sagemaker"

common_env = {
    "SGLANG_DISABLE_CUDNN_CHECK": "1",
    "SM_SGLANG_ATTENTION_BACKEND": "triton",
    "SM_SGLANG_MEM_FRACTION_STATIC": "0.8",
    "SM_SGLANG_FP8_GEMM_BACKEND": "triton",
    "SM_SGLANG_MOE_RUNNER_BACKEND": "triton",
    "SM_SGLANG_MODEL_PATH": model_id
}
sgl_env = {
    "SM_SGLANG_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_SGLANG_CONTEXT_LENGTH": "262144",
    "SM_SGLANG_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_SGLANG_REASONING_PARSER": "qwen3",
    "SM_SGLANG_SPECULATIVE_ALGORITHM": "EAGLE",
    "SM_SGLANG_SPECULATIVE_NUM_STEPS": "3",
    "SM_SGLANG_SPECULATIVE_EAGLE_TOPK": "1",
    "SM_SGLANG_SPECULATIVE_NUM_DRAFT_TOKENS": "4",
}
env = common_env | sgl_env

### Option 3: LMI

In [ ]:
CONTAINER_VERSION = "0.38.0-lmi30.0.0-cu130"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

spec_config = {
    "method": "mtp",
    "num_speculative_tokens": 3
}
common_env = {
    "HF_MODEL_ID": model_id,
}
lmi_env = {
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": json.dumps(instance["num_gpu"]),
    "OPTION_MAX_MODEL_LEN": "262144",
    "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
    "OPTION_TOOL_CALL_PARSER": "qwen3_coder",
    "OPTION_REASONING_PARSER": "qwen3",
    "OPTION_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | lmi_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

---
## Inference examples

### Best Practices

We recommend using the following set of sampling parameters for generation

- Thinking mode: temperature=1.0, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0

- Instruct (or non-thinking) mode: temperature=0.7, top_p=0.80, top_k=20, min_p=0.0, presence_penalty=1.5, repetition_penalty=1.0

Please note that the support for sampling parameters varies according to inference frameworks.

Qwen3.8 models operate in thinking mode by default, generating thinking content signified by <think>\n...</think>\n\n before producing the final responses. Reasoning depth can be tuned with `reasoning_effort` (`xhigh` by default, `medium`, `low`), passed through `chat_template_kwargs`. To disable thinking and obtain a direct response, set `"chat_template_kwargs": {"enable_thinking": false}` (see the examples below).

---
#### Text inference

In [ ]:
payload={
    "messages": [
        {"role": "user", "content": "Name popular places to visit in London?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 2.52s





Here are some popular places to visit in London:

- Tower of London  
- Buckingham Palace  
- The British Museum  
- London Eye  
- Tower Bridge  
- Westminster Abbey  
- Big Ben and the Houses of Parliament  
- London Bridge  
- Trafalgar Square  
- Hyde Park  
- The Natural History Museum  
- Covent Garden  
- Shard London Bridge  
- Royal Albert Hall  
- South Bank  
- Notting Hill  
- Borough Market  
- Camden Market  
- Greenwich Park and the Royal Observatory  
- The Churchill War Rooms

-----------------------
{'prompt_tokens': 60, 'total_tokens': 206, 'completion_tokens': 146, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 33}}


---
#### Video Understanding



<video src="https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/video/N1cdUjctpG8.mp4">

In [ ]:
payload={
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/video/N1cdUjctpG8.mp4"
                    }
                },
                {
                    "type": "text",
                    "text": "How many porcelain jars were discovered in the niches located in the primary chamber of the tomb?"
                }
            ]
        }
    ],
    "top_k": 20,
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 17.16s





**Nine porcelain jars.** The photo of the interior shows a row of eight jars (alternating larger and smaller vessels) lined up along the brick‑walled niche, plus one additional smaller jar set in front, for a total of nine jars.

-----------------------
{'prompt_tokens': 11157, 'total_tokens': 12265, 'completion_tokens': 1108, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 1055}}


---
#### Text generation with reasoning output

In [ ]:
payload={
    "messages": [
        {"role": "user", "content": "What is the derivative of x^3 * ln(x)?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
message = response["choices"][0]["message"]
# vLLM / LMI return the reasoning trace in "reasoning", SGLang in "reasoning_content"
reasoning = message.get("reasoning") or message.get("reasoning_content")
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
display(Markdown('### Content:\n---'))
display(Markdown(message["content"]))
display(Markdown('---'))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 2.29s



### Reasoning:
---

We need answer user's math question: derivative of x^3 * ln(x). Need provide final. Compute derivative: product rule: d/dx x^3 ln x = 3x^2 ln x + x^3*(1/x)=3x^2 ln x + x^2 = x^2(3 ln x +1), assuming x>0 for ln real. Could also mention for x<0 complex? standard domain x>0. Final concise.


### Content:
---



Using the product rule:

\[
\frac{d}{dx}\left(x^3 \ln x\right)
= 3x^2 \ln x + x^3 \cdot \frac{1}{x}
\]

So,

\[
\frac{d}{dx}\left(x^3 \ln x\right)
= 3x^2 \ln x + x^2
\]

or factored:

\[
\boxed{x^2(3\ln x + 1)}
\]

---

-----------------------
{'prompt_tokens': 64, 'total_tokens': 282, 'completion_tokens': 218, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 99}}


---
#### Controlling reasoning depth with `reasoning_effort`

Qwen3.8's chat template accepts `reasoning_effort` (`xhigh` default, `medium`, `low`). Lower effort produces shorter reasoning and lower latency; in multi-turn agentic tasks it can also lead to more retries, so measure end-to-end task cost before lowering it.

In [ ]:
question = ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
            "Five bats and three balls are bought with a 10% discount on the bats only. What is the total cost?")

for effort in ["low", "medium", "xhigh"]:
    payload = {
        "messages": [{"role": "user", "content": question}],
        "chat_template_kwargs": {"reasoning_effort": effort},
    }
    start_time = time.time()
    res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                     Body=json.dumps(payload),
                                     ContentType="application/json")
    response = json.loads(res["Body"].read().decode("utf8"))
    end_time = time.time()
    usage = response["usage"]
    # vLLM / LMI report reasoning tokens in completion_tokens_details, SGLang at the top level of usage
    reasoning_tokens = (usage.get("completion_tokens_details") or {}).get("reasoning_tokens", usage.get("reasoning_tokens", "n/a"))
    print(f"reasoning_effort={effort:<6} | response time: {end_time-start_time:6.2f}s | "
          f"completion tokens: {usage['completion_tokens']:>5} | reasoning tokens: {reasoning_tokens}")

reasoning_effort=low    | response time:   5.03s | completion tokens:   549 | reasoning tokens: 328


reasoning_effort=medium | response time:   5.60s | completion tokens:   599 | reasoning tokens: 329


reasoning_effort=xhigh  | response time:   7.00s | completion tokens:   652 | reasoning tokens: 387


---
#### Instruct (non-thinking) mode

In [ ]:
payload={
    "messages": [
        {"role": "user", "content": "Give me a one-sentence summary of what Amazon SageMaker AI is."}
    ],
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "presence_penalty": 1.5,
    "chat_template_kwargs": {"enable_thinking": False},
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 0.52s



Amazon SageMaker AI is a fully managed platform that provides tools for building, training, and deploying machine learning models at scale.

-----------------------
{'prompt_tokens': 27, 'total_tokens': 53, 'completion_tokens': 26, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 0}}


---
#### Tool calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. London"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather like in Paris right now, in celsius?"}]

# Turn 1: the model decides to call the tool
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps({"messages": messages, "tools": tools, "tool_choice": "auto"}),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
assistant_msg = response["choices"][0]["message"]
tool_call = assistant_msg["tool_calls"][0]
print("Tool call:", tool_call["function"]["name"], tool_call["function"]["arguments"])

# Turn 2: return a (mock) tool result and let the model answer
messages += [
    {"role": "assistant", "content": assistant_msg.get("content") or "", "tool_calls": assistant_msg["tool_calls"]},
    {"role": "tool", "tool_call_id": tool_call["id"],
     "content": json.dumps({"city": "Paris", "temperature": 18, "unit": "celsius", "condition": "partly cloudy"})},
]
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps({"messages": messages, "tools": tools}),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
display(Markdown(response["choices"][0]["message"]["content"]))

Tool call: get_current_weather {"city": "Paris", "unit": "celsius"}




It's currently **18°C** in Paris, with partly cloudy skies.

---
#### Streaming

In [ ]:
#
# Helper function for streaming invocation
#
import io
import json
import time
import boto3
from IPython.display import clear_output

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:", chunk)
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def stream_response(endpoint_name, inputs, max_tokens=8189, temperature=1.0, top_p=0.95):    
    body = {
      "messages": [
            {"role": "user", "content": [{"type": "text", "text": inputs}]}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "stream": True,
        "stream_options": {"include_usage": True},
        "chat_template_kwargs": {"enable_thinking": True}
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName = endpoint_name,
        Body = json.dumps(body),
        ContentType = "application/json",
    )

    event_stream = resp["Body"]
    start_json = b"{"
    full_response = ""
    start_time = time.time()
    token_count = 0
    tps = 0
    usage = None

    for line in LineIterator(event_stream):
        if line != b"" and start_json in line:
            data = json.loads(line[line.find(start_json):].decode("utf-8"))
            if data.get("usage"):
                usage = data["usage"]
            if not data.get("choices"):
                continue
            delta = data['choices'][0]['delta']
            # vLLM / LMI stream reasoning as "reasoning", SGLang as "reasoning_content"
            token_text = (delta.get('reasoning') or delta.get('reasoning_content') or '') + (delta.get('content') or '')
            if not token_text:
                continue
            full_response += token_text
            token_count += 1

            # Clear the output and reprint everything
            clear_output(wait=True)
            display(Markdown(full_response))

    elapsed_time = time.time() - start_time
    # With speculative decoding one chunk can carry several tokens, so use the server-side token count
    completion_tokens = usage["completion_tokens"] if usage else token_count
    tps = completion_tokens / elapsed_time
    print(f"\nCompletion tokens: {completion_tokens}, elapsed: {elapsed_time:.2f}s, Tokens per Second: {tps:.2f}\n")

In [ ]:
#inputs = "What is greater 9.11 or 9.8?"
inputs = "Solve this problem step by step: What is 15% of 240?"
output = stream_response(endpoint_name, inputs, max_tokens=8000)

We need answer simple math. Need provide step-by-step. Then final. Compute 15% of 240 = 0.15*240=36. Could show 10% +5%.


Step 1: Convert 15% to a decimal:  
15% = 0.15

Step 2: Multiply by 240:  
0.15 × 240 = 36

Answer: **36**


Completion tokens: 108, elapsed: 1.04s, Tokens per Second: 103.61



---
#### Image understanding

![image](https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/CI_Demo/mathv-1327.jpg)

In [ ]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/CI_Demo/mathv-1327.jpg"
                    }
                },
                {
                    "type": "text",
                    "text": "The centres of the four illustrated circles are in the corners of the square. The two big circles touch each other and also the two little circles. With which factor do you have to multiply the radii of the little circles to obtain the radius of the big circles?\nChoices:\n(A) $\\frac{2}{9}$\n(B) $\\sqrt{5}$\n(C) $0.8 \\cdot \\pi$\n(D) 2.5\n(E) $1+\\sqrt{2}$"
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 29.70s





Let the square have side $a$, the big radius be $R$ and the small radius $r$. Place the big circles at opposite corners and the small circles at the other two corners.

**Big circles touch each other** → their centres lie on a diagonal, so
$$2R = a\sqrt{2} \implies R = \frac{a}{\sqrt{2}}$$

**A big circle touches a small circle** (adjacent corners, distance $a$) →
$$R + r = a \implies r = a - \frac{a}{\sqrt{2}} = a\left(1 - \frac{1}{\sqrt{2}}\right)$$

Therefore
$$\frac{R}{r} = \frac{a/\sqrt{2}}{a\left(1 - 1/\sqrt{2}\right)} = \frac{1}{\sqrt{2}-1} = \sqrt{2}+1$$

So the little radii must be multiplied by $1+\sqrt{2}$.

**Answer: (E)**

-----------------------
{'prompt_tokens': 313, 'total_tokens': 3035, 'completion_tokens': 2722, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 2486}}


---
### Performance improvement by using speculative decoding

![SD vs no-SD Performance](sd_vs_nosd.png)

## Key Takeaways

Measured with the vLLM option (vLLM 0.32.0, MTP with 3 speculative tokens) on `ml.g6e.24xlarge`, non-thinking mode, `temperature=0`, 512 output tokens per request:

| Metric | Concurrency | SD | no-SD | SD Improvement |
|---|---|---|---|---|
| **Latency** | 1 | 5,514 ms | 11,632 ms | **−53%** |
| **Latency** | 10 | 10,300 ms | 18,225 ms | **−43%** |
| **Throughput** | 1 | 92.8 tokens/sec | 44.0 tokens/sec | **+111%** |
| **Throughput** | 10 | 491.7 tokens/sec | 280.3 tokens/sec | **+75%** |

### Summary

- **SD outperforms no-SD** on both latency and throughput at both concurrency levels.
- **Largest gain at concurrency 1**: SD cuts average request latency by **53%** (11,632 ms to 5,514 ms) and more than doubles tokens/sec (93 vs 44).
- At **concurrency 10**, SD still gives a **43% latency reduction** and **75% higher throughput**.
- vLLM reported a mean MTP acceptance length of about 2.98 tokens per decoding step during this run. Gains depend on prompt, sampling settings, and output length.


## Cleanup

In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)